[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sapiosciences/sapio-py-tutorials/blob/master/10_python_record_models.ipynb)

In [ ]:
from sapiopylib.rest.DataMgmtService import DataMgmtServer
from sapiopylib.rest.DataRecordManagerService import DataRecordManager
from sapiopylib.rest.User import SapioUser
from sapiopylib.rest.utils.recordmodel.PyRecordModel import PyRecordModel
from sapiopylib.rest.utils.recordmodel.RecordModelManager import RecordModelManager, RecordModelInstanceManager, \
    RecordModelRelationshipManager

user = SapioUser(url="https://linux-vm:8443/webservice/api", verify_ssl_cert=False,
                 guid="3c232543-f407-4828-aae5-b33d4cd31fa7",
                 username="yqiao_api", password="Password1!")
dataRecordManager: DataRecordManager = DataMgmtServer.get_data_record_manager(user)

# Record Model Basics
Record models are an established component of the Sapio Commons library. This module supports Sapio plugin and script development: developers no longer need to manually manage request batching, and they can maintain local state to simplify record manipulation.

The Python Record Model module shares the design goals of its Java counterpart. It enables developers to:


*   Create new records.
*   Manipulate existing records.
*   Create and remove parent-child links.
*   Maintain a client-side record cache of all records and links in the Python client.
*   Support custom event handlers that fire on record creation, modification, link modification, store, or rollback.

Let's get started! First, we instantiate Python record models.

In [ ]:
# Used to store and commit changes made to record models, also holds other managers
rec_man: RecordModelManager = RecordModelManager(user)

# Used to create new records and return the corresponding record models and convert data records to record models
inst_man: RecordModelInstanceManager = rec_man.instance_manager

# Used to modify parent/child relationships between records using record models
relationship_man: RecordModelRelationshipManager = rec_man.relationship_manager

It is important to understand how this instance was constructed.

The Python record model is a caching system, so it is important to maintain the user's cache without deleting it, for both data integrity and performance.

When this call is executed:
```
rec_man: RecordModelManager = RecordModelManager(user)
```
The module first checks whether a record model manager has already been created for this User instance. If one exists, the module does not build a new Record Model Manager, and instead retrieves the existing one tied to this User instance.
In other words, the record model cache is a singleton per user context.

The following example demonstrates this caching behavior in practice:

In [ ]:
sample_record = inst_man.add_new_record('Sample')
sample_record.set_field_value('OtherSampleId', 'PyRecordModel Test')
sample_record.set_field_value('ExemplarSampleType', 'DNA')
print(rec_man)
print(sample_record)

rec_man: RecordModelManager = RecordModelManager(user)
inst_man: RecordModelInstanceManager = rec_man.instance_manager
sample_record = inst_man.get_known_record_with_record_id(sample_record.record_id)
print(rec_man)
print(sample_record)

As demonstrated above, creating a new RecordModelManager instance returns the existing singleton cache reference for this user context.

You may also notice that the created sample record has not yet been written to Sapio. If you try to search for this new sample, it does not exist. This is intentional: for performance reasons, the user must explicitly call a "store and commit" method on the record model manager before the data is transferred to Sapio.

Before running this example, create a location called "Default Location" and give it a child department with any name other than "Mice Research".
Let's complete this entire example for the first round. We search for a location named "Default Location" and rename it to "My Location".
At the same time, we create a new Department record named "Mice Research" and insert this sample, plus another sample, under "Mice Research".

In [ ]:
# Query the location named 'Default Location' (assumed to exist; error handling omitted for brevity).
default_location_record = dataRecordManager.query_data_records('VeloxLocation',
                                                               'LocationName', ['Default Location']).result_list[0]

# Wrap this DataRecord in a record model using add_existing_record.
default_location: PyRecordModel = inst_man.add_existing_record(default_location_record)

# Update the location name on the record model. Notice that the backing DataRecord remains unchanged until committed.
default_location.set_field_value('LocationName', 'My Location')
print("Backing location record's location: " + default_location.get_data_record().get_field_value('LocationName'))
print("Record model's location name: " + default_location.get_field_value('LocationName'))

# Create a new department record model directly without an existing DataRecord backing it.
mice_research_dept: PyRecordModel = inst_man.add_new_record('VeloxDepartment')
mice_research_dept.set_field_value('DepartmentName', 'Mice Research')

# Add this new department as a child record to the location in our record model cache.
default_location.add_child(mice_research_dept)

mice_research_dept.add_child(sample_record)

sample_record2 = inst_man.add_new_record('Sample')
sample_record2.set_field_value('OtherSampleId', 'PyRecordModel Test')
sample_record2.set_field_value('ExemplarSampleType', 'Urine')

mice_research_dept.add_child(sample_record2)

### Loading Record Relationships
You can only access a record's relationships if you have already loaded them using the relationship manager, or if the record is newly created (or deleted).

Attempting to access an existing record's relationships without first calling the load method raises an exception. For newly created records, however, relationships can be accessed without explicit loading.

Parent-child relationships are bidirectional, allowing queries in either direction for both new and existing relationships.

When relationships are added to existing ones, the cached view reflects all staged changes in the resulting list.

In [ ]:
from typing import List
from sapiopylib.rest.utils.recordmodel.PyRecordModel import SapioRecordModelException

try:
    default_location.get_children_of_type('VeloxDepartment')
except SapioRecordModelException as e:
    print(str(e))

relationship_man.load_children([default_location], 'VeloxDepartment')
print("Number of departments under Default Location: " + str(len(default_location.get_children_of_type('VeloxDepartment'))))

samples_under_dept: List[PyRecordModel] = mice_research_dept.get_children_of_type('Sample')
print("Number of samples under new Department: " +
      str(len(samples_under_dept)))

departments_above_sample: List[PyRecordModel] = sample_record.get_parents_of_type('VeloxDepartment')
print("Number of departments above Sample: " +
      str(len(departments_above_sample)))

Record Model with Record ID 90: Child type VeloxDepartment was not loaded.
Number of departments under Default Location: 2
Number of samples under new Department: 2
Number of departments above Sample: 1


Finally, we can store the changes to make them permanent.

In [ ]:
rec_man.store_and_commit()

# Record Model with Wrappers

Record model wrappers decorate the root record models with syntactic sugar. These wrapper classes provide getters and setters named after the fields and data types, which makes the code much easier to learn, read, and use. This is documentation as code.

Record model wrappers can be generated in your Sapio system.
In the Sapio web client, click App Setup under your profile icon, then select Manage Rules & Webhooks -> Webservice API from the left menu.

Click the "Generate Python Record Model Wrappers for System Data Types" button.

![Generate Python Record Model Wrappers button](resources/generate_record_model_wrappers.png)

The platform finds all data types in the system and generates a Python record model class for each one. The generated Python code is then downloaded through your browser. The file is always named "data_type_models.py", but you can rename it after you receive it. Note that the file name becomes the module name when the file is imported into a Python script.

Here is an example of how to use the wrappers:

In [ ]:
from sapiopylib.rest.DataMgmtService import DataMgmtServer
from sapiopylib.rest.DataRecordManagerService import DataRecordManager
from sapiopylib.rest.User import SapioUser
from sapiopylib.rest.utils.recordmodel.RecordModelManager import RecordModelManager

from data_type_models import SampleModel

# Creating New Wrapped Record Models
user = SapioUser(url="https://linux-vm:8443/webservice/api", verify_ssl_cert=False,
                 guid="3c232543-f407-4828-aae5-b33d4cd31fa7",
                 username="yqiao_api", password="Password1!")
dataRecordManager: DataRecordManager = DataMgmtServer.get_data_record_manager(user)

rec_man = RecordModelManager(user)
inst_man = rec_man.instance_manager
relationship_man = rec_man.relationship_manager

# Whenever we need to fill a WrapperType generic variable as hinted by IDE, we can use the type of any wrapped models
# As of the latest release, PyCharm may not provide correct typing hints without an explicit declaration of the WrapperType return value.
# Therefore, it can be important to declare ": SampleModel" here for PyCharm.
parent_sample: SampleModel = inst_man.add_new_record_of_type(SampleModel)
child_sample: SampleModel = inst_man.add_new_record_of_type(SampleModel)

parent_sample.add_child(child_sample)

parent_sample.set_OtherSampleId_field("Wrapped Record Model Testing Parent")
child_sample.set_OtherSampleId_field("Wrapped Record Model Testing Child")

rec_man.store_and_commit()

user = SapioUser(url="https://linux-vm:8443/webservice/api", verify_ssl_cert=False,
                 guid="3c232543-f407-4828-aae5-b33d4cd31fa7",
                 username="yqiao_api", password="Password1!")
dataRecordManager: DataRecordManager = DataMgmtServer.get_data_record_manager(user)

# Reading existing record models
rec_man = RecordModelManager(user)
inst_man = rec_man.instance_manager
relationship_man = rec_man.relationship_manager

parent_sample = inst_man.add_existing_record_of_type(parent_sample.get_data_record(), SampleModel)

relationship_man.load_children_of_type([parent_sample], SampleModel)
children_samples = parent_sample.get_children_of_type(SampleModel)

print([str(x) for x in children_samples])

# Record Model Properties
To provide a consistent programming interface between wrapped and unwrapped record models, and to be consistent with our Java Record Model API, we have implemented record model properties on both the WrappedRecordModel and PyRecordModel classes. Some properties require the appropriate load() method to be called before use.

The following types of record model properties are available:


1.   Getter properties: obtain the current value of a property of the record model.
2.   Adder properties: if the property is an iterable, add new items to the iterable of the specified property of the record model.
3.   Remover properties: if the property is an iterable, remove items from the specified property of the record model.
4.   Setter properties: set the specified property of the record model to a new value, overwriting any existing value.

Below is an example demonstrating record model properties:

In [ ]:
from sapiopylib.rest.utils.recordmodel.properties import *

parent_sample: PyRecordModel = inst_man.add_new_record("Sample")
children_samples = parent_sample.add(Children.create_by_name("Sample", 5))
children_samples_test = parent_sample.get(Children.of_type_name("Sample"))

parent_sample.remove(Children.refs(children_samples))

parent_sample.add(Children.refs(children_samples))

You may also use the wrapped record model forms:

In [ ]:
parent_sample: SampleModel = inst_man.add_new_record_of_type(SampleModel)
child_sample: SampleModel = parent_sample.add(Child.create(SampleModel))
child_sample_test: SampleModel = parent_sample.get(Child.of_type(SampleModel))